# Détection d'anomalies CAN - VAE avec RNN

Projet CANlock - Session H26

**Architecture:** Variational Auto-Encoder (VAE) avec LSTM bidirectionnel en encodeur et décodeur.  
**Principe:** L'encodeur apprend les paramètres (mu, sigma) d'une distribution latente à partir des séquences normales. Le décodeur reconstruit à partir d'un échantillon de cette distribution. Les anomalies, hors distribution, sont mal reconstruites.

**Données de test:** Attaques réalistes générées par le framework `attack_synthesis` (spoofing, masquerade, replay, suspension).

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))

import warnings
warnings.filterwarnings('ignore')

In [2]:
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, precision_recall_curve
import pytorch_lightning as pl
from pytorch_lightning.callbacks import EarlyStopping, ModelCheckpoint
from pytorch_lightning.loggers import MLFlowLogger
from tqdm import tqdm

from canlock.db.database import get_session, init_db
from canlock.db.models import CanMessage, PgnDefinition, SpnDefinition
from canlock.decoder import SessionDecoder
from canlock.data.can_data_module import CANDataModule
from canlock.models.rnn_vae import RnnVae
from canlock.attacks.spoofing_attack import SpoofingAttack
from canlock.attacks.masquerade_attack import MasqueradeAttack
from canlock.attacks.replay_attack import ReplayAttack
from canlock.attacks.suspension_attack import SuspensionAttack

from sqlmodel import select

pl.seed_everything(42)
device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
print(f"Device: {device}")

Seed set to 42


Device: mps


## Chargement des données brutes depuis la DB

In [3]:
init_db()

# Charger les messages bruts CAN
LIMIT = 50000

with get_session() as session:
    rows = session.exec(
        select(CanMessage).order_by(CanMessage.timestamp).limit(LIMIT)
    ).all()

df_raw_messages = pd.DataFrame([{
    "timestamp": r.timestamp,
    "can_identifier": r.can_identifier,
    "length": r.length,
    "payload": r.payload,
} for r in rows])

print(f"Messages bruts chargés: {len(df_raw_messages):,}")

Messages bruts chargés: 50,000


## Génération des attaques avec le framework attack_synthesis

On applique les 4 types d'attaques réalistes de Loïc sur les messages bruts:
- **Spoofing**: injection/remplacement de payloads avec bruit gaussien
- **Masquerade**: usurpation d'adresse source ECU
- **Replay**: retransmission de trames valides avec délai
- **Suspension**: simulation de bus-off (suppression de trames)

In [4]:
# Appliquer les attaques sur une copie des messages bruts
df_attacked = df_raw_messages.copy()

# Instancier les attaques
spoof = SpoofingAttack(injection_rate=0.03, mode="replace")
masq = MasqueradeAttack(attacker_source=0x99, prob=0.02)
replay = ReplayAttack(replay_rate=0.03, delay_seconds=1.0)
susp = SuspensionAttack(suspend_fraction=0.02)

# Application séquentielle
df_attacked = spoof.apply(df_attacked, target=None)
df_attacked = masq.apply(df_attacked, target=None)
df_attacked = replay.apply(df_attacked, target=None)
df_attacked = susp.apply(df_attacked, target=None)

# Trier par timestamp
df_attacked = df_attacked.sort_values("timestamp").reset_index(drop=True)

print(f"Messages après attaques: {len(df_attacked):,}")
print(f"\nDistribution des attaques:")
print(df_attacked["attack_type"].value_counts())

Messages après attaques: 51,500

Distribution des attaques:
attack_type
replay        1472
spoofing      1432
suspension    1030
masquerade    1021
Name: count, dtype: int64


## Décodage des messages → valeurs SPN

On décode les messages bruts (normaux et attaqués) en valeurs SPN exploitables par le modèle.

In [5]:
def decode_raw_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    """
    Décode un DataFrame de messages CAN bruts en valeurs SPN.
    Préserve la colonne attack_type si présente.
    """
    with get_session() as session:
        decoder = SessionDecoder(db=session)
        
        # Construire le cache PGN/SPN
        pgn_cache = {}
        decoded_rows = []
        
        for _, msg in tqdm(df.iterrows(), total=len(df), desc="Décodage"):
            if msg["can_identifier"] is None or msg["payload"] is None:
                continue
            if pd.isna(msg["can_identifier"]) or pd.isna(msg.get("payload", None)):
                continue
                
            try:
                payload = bytes(msg["payload"]) if not isinstance(msg["payload"], bytes) else msg["payload"]
            except (TypeError, ValueError):
                continue
                
            pgn_number = SessionDecoder.extract_pgn_number_from_payload(int(msg["can_identifier"]))
            
            if pgn_number not in pgn_cache:
                pgn_def = session.exec(
                    select(PgnDefinition).where(PgnDefinition.pgn_identifier == pgn_number)
                ).first()
                
                if pgn_def:
                    spns = session.exec(
                        select(SpnDefinition).where(SpnDefinition.pgn_id == pgn_def.id)
                    ).all()
                    pgn_cache[pgn_number] = [(spn, spn.analog_attributes) for spn in spns]
                else:
                    pgn_cache[pgn_number] = None
            
            spn_rules = pgn_cache[pgn_number]
            if spn_rules:
                spns = [s[0] for s in spn_rules]
                analog_attrs = [s[1] for s in spn_rules]
                values = decoder.extract_values_from_spns(spns, analog_attrs, payload)
                if values:
                    row = {"timestamp": msg["timestamp"]}
                    row.update(values)
                    if "attack_type" in msg:
                        row["attack_type"] = msg["attack_type"]
                    decoded_rows.append(row)
    
    return pd.DataFrame(decoded_rows)

In [6]:
# Décoder les messages normaux (originaux)
df_raw_messages["attack_type"] = "normal"
df_normal_decoded = decode_raw_dataframe(df_raw_messages)
print(f"Messages normaux décodés: {len(df_normal_decoded)}")

# Décoder les messages attaqués
df_attacked_decoded = decode_raw_dataframe(df_attacked)
print(f"Messages attaqués décodés: {len(df_attacked_decoded)}")
print(f"\nDistribution après décodage:")
print(df_attacked_decoded["attack_type"].value_counts())

Décodage:   0%|          | 0/50000 [00:00<?, ?it/s]

Décodage:   0%|          | 12/50000 [00:00<06:58, 119.53it/s]

Décodage:   0%|          | 103/50000 [00:00<01:25, 583.66it/s]

Décodage:   2%|▏         | 1248/50000 [00:00<00:08, 5540.36it/s]

Décodage:   9%|▊         | 4272/50000 [00:00<00:02, 15283.61it/s]

...

Décodage:  82%|████████▏ | 42350/51500 [00:01<00:00, 29053.83it/s]

Décodage:  88%|████████▊ | 45532/51500 [00:01<00:00, 29839.12it/s]

Décodage:  95%|█████████▍| 48739/51500 [00:01<00:00, 30481.36it/s]

Décodage: 100%|██████████| 51500/51500 [00:02<00:00, 25273.47it/s]

Messages attaqués décodés: 40334

Distribution après décodage:
attack_type
replay        1161
spoofing      1148
masquerade     813
Name: count, dtype: int64


## Préparation des données pour le VAE

1. Sélectionner les SPN communs entre normal et attaqué
2. Normaliser avec StandardScaler
3. Créer des séquences temporelles
4. Labelliser: normal=0, attaque=1

In [7]:
# Colonnes SPN communes (exclure timestamp et attack_type)
meta_cols = {"timestamp", "attack_type"}
normal_spn_cols = set(df_normal_decoded.columns) - meta_cols
attacked_spn_cols = set(df_attacked_decoded.columns) - meta_cols
common_cols = sorted(normal_spn_cols & attacked_spn_cols)

# Garder les top colonnes par fill rate
fill_rates = df_normal_decoded[common_cols].notna().mean().sort_values(ascending=False)
top_cols = fill_rates.head(15).index.tolist()
print(f"Colonnes SPN sélectionnées: {len(top_cols)}")
print(top_cols)

Colonnes SPN sélectionnées: 15
['MANUFACTURER_DEFINED_USAGE_PROPB_PDU2', 'STEERING_WHEEL_ANGLE', 'YAW_RATE', 'LONGITUDINAL_ACCELERATION', 'LATERAL_ACCELERATION', 'STEERING_WHEEL_ANGLE_SENSOR_TYPE', 'STEERING_WHEEL_TURN_COUNTER', 'MOMENTARY_ENGINE_MAXIMUM_POWER_ENABLE', 'PERCENT_CLUTCH_SLIP', 'TRANSMISSION_1_OUTPUT_SHAFT_SPEED', 'TRANSMISSION_DRIVELINE_ENGAGED', 'TRANSMISSION_1_INPUT_SHAFT_SPEED', 'PROGRESSIVE_SHIFT_DISABLE', 'ENGINE_MOMENTARY_OVERSPEED_ENABLE', 'SOURCE_ADDRESS_OF_CONTROLLING_DEVICE_FOR_TRANSMISSION_CONTROL']


In [8]:
# Préparer les données normales (pour l'entraînement)
df_normal_clean = df_normal_decoded[top_cols].ffill().bfill().dropna()
print(f"Données normales nettoyées: {df_normal_clean.shape}")

# Préparer les données attaquées (pour le test)
df_attacked_clean = df_attacked_decoded[top_cols + ["attack_type"]].copy()
df_attacked_clean[top_cols] = df_attacked_clean[top_cols].ffill().bfill()
df_attacked_clean = df_attacked_clean.dropna(subset=top_cols)
print(f"Données attaquées nettoyées: {df_attacked_clean.shape}")

Données normales nettoyées: (39973, 15)
Données attaquées nettoyées: (40334, 16)


In [9]:
# Normalisation (fit sur les données normales uniquement)
scaler = StandardScaler()
normal_scaled = scaler.fit_transform(df_normal_clean.values)
attacked_scaled = scaler.transform(df_attacked_clean[top_cols].values)
attacked_labels = (df_attacked_clean["attack_type"] != "normal").astype(int).values

# Créer les séquences
SEQ_LEN = 30
STRIDE = 30

def create_sequences(data, seq_len, stride):
    sequences = []
    for i in range(0, len(data) - seq_len + 1, stride):
        sequences.append(data[i:i + seq_len])
    return np.array(sequences)

def create_sequences_with_labels(data, labels, seq_len, stride):
    sequences = []
    seq_labels = []
    for i in range(0, len(data) - seq_len + 1, stride):
        sequences.append(data[i:i + seq_len])
        # Une séquence est anomale si elle contient au moins un message attaqué
        seq_labels.append(1 if labels[i:i + seq_len].sum() > 0 else 0)
    return np.array(sequences), np.array(seq_labels, dtype=np.int64)

# Séquences normales (entraînement)
normal_sequences = create_sequences(normal_scaled, SEQ_LEN, STRIDE)
normal_labels = np.zeros(len(normal_sequences), dtype=np.int64)

# Séquences attaquées (test)
attacked_sequences, attacked_seq_labels = create_sequences_with_labels(
    attacked_scaled, attacked_labels, SEQ_LEN, STRIDE
)

print(f"Séquences normales: {normal_sequences.shape}")
print(f"Séquences attaquées: {attacked_sequences.shape}")
print(f"  - Normales dans set attaqué: {(attacked_seq_labels == 0).sum()}")
print(f"  - Anomalies dans set attaqué: {(attacked_seq_labels == 1).sum()}")

Séquences normales: (1332, 30, 15)
Séquences attaquées: (1344, 30, 15)
  - Normales dans set attaqué: 0
  - Anomalies dans set attaqué: 1344


In [10]:
# Split en 3 sets:
# - Train: normales (pour l'entraînement)
# - Val: normales (pour monitorer que l'apprentissage va dans le bon sens)
# - Test: normales + attaques (pour évaluer la détection)

# D'abord split les normales: 60% train, 20% val, 20% test
X_train_normal, X_temp, y_train_normal, y_temp = train_test_split(
    normal_sequences, normal_labels, test_size=0.4, random_state=42
)
X_val, X_test_normal, y_val, y_test_normal = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42
)

# Test set = normales (20%) + toutes les attaques
X_test = np.concatenate([X_test_normal, attacked_sequences], axis=0)
y_test = np.concatenate([y_test_normal, attacked_seq_labels], axis=0)

print(f"Train: {len(X_train_normal)} séquences normales")
print(f"Val: {len(X_val)} séquences normales")
print(f"Test: {len(X_test)} séquences ({(y_test==0).sum()} normales, {(y_test==1).sum()} anomalies)")

Train: 799 séquences normales
Val: 266 séquences normales
Test: 1611 séquences (267 normales, 1344 anomalies)


## Modèle VAE-RNN

L'architecture utilise:
- **Encodeur**: LSTM bidirectionnel → paramètres (mu, logvar) de la distribution latente
- **Reparameterization trick**: z = mu + sigma * epsilon (permet la backpropagation)
- **Décodeur**: LSTM bidirectionnel → reconstruction de la séquence
- **Loss**: MSE (reconstruction) + KL divergence (régularisation de l'espace latent)

In [11]:
# Hyperparamètres
BATCH_SIZE = 32
MAX_EPOCHS = 50
LR = 0.001
HIDDEN_DIM = 64
LATENT_DIM = 32
N_LAYERS = 1
KL_WEIGHT = 0.001

# DataModule avec validation set séparé (normales uniquement)
datamodule = CANDataModule(
    X_train_normal, X_test,
    y_train_normal, y_test,
    batch_size=BATCH_SIZE,
    X_val=X_val, y_val=y_val,
)

# Modèle
n_features = X_train_normal.shape[2]
seq_len = X_train_normal.shape[1]

model = RnnVae(
    n_features=n_features,
    seq_len=seq_len,
    hidden_dim=HIDDEN_DIM,
    latent_dim=LATENT_DIM,
    n_layers=N_LAYERS,
    lr=LR,
    kl_weight=KL_WEIGHT,
)
print(model)

n_params = sum(p.numel() for p in model.parameters())
print(f"\nParamètres: {n_params:,}")

RnnVae(
  (encoder_lstm): LSTM(15, 64, batch_first=True, bidirectional=True)
  (encoder_dropout): Dropout(p=0.2, inplace=False)
  (fc_mu): Linear(in_features=128, out_features=32, bias=True)
  (fc_logvar): Linear(in_features=128, out_features=32, bias=True)
  (latent_to_hidden): Linear(in_features=32, out_features=128, bias=True)
  (decoder_lstm): LSTM(128, 64, batch_first=True, bidirectional=True)
  (decoder_dropout): Dropout(p=0.2, inplace=False)
  (output_layer): Linear(in_features=128, out_features=15, bias=True)
)

Paramètres: 155,215


## Entraînement avec PyTorch Lightning
- Suivi des expérimentations avec MLflow
- Early stopping sur val_loss
- Métriques loggées: loss total, reconstruction loss, KL divergence

In [12]:
# MLflow Logger
mlflow_logger = MLFlowLogger(
    experiment_name="rnn_vae",
    tracking_uri="http://localhost:5050",
    log_model=True,
)

# Callbacks
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=10,
    mode='min',
    verbose=True
)

checkpoint = ModelCheckpoint(
    dirpath='../models/',
    filename='rnn_vae-{epoch:02d}-{val_loss:.4f}',
    monitor='val_loss',
    mode='min',
    save_top_k=1
)

# Trainer
trainer = pl.Trainer(
    max_epochs=MAX_EPOCHS,
    accelerator='auto',
    devices=1,
    callbacks=[early_stop, checkpoint],
    enable_progress_bar=True,
    log_every_n_steps=10,
    logger=mlflow_logger,
)

# Entraînement
trainer.fit(model, datamodule)

GPU available: True (mps), used: True


TPU available: False, using: 0 TPU cores


💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.



  | Name             | Type    | Params | Mode  | FLOPs
-------------------------------------------------------------
0 | encoder_lstm     | LSTM    | 41.5 K | train | 0    
1 | encoder_dropout  | Dropout | 0      | train | 0    
2 | fc_mu            | Linear  | 4.1 K  | train | 0    
3 | fc_logvar        | Linear  | 4.1 K  | train | 0    
4 | latent_to_hidden | Linear  | 4.2 K  | train | 0    
5 | decoder_lstm     | LSTM    | 99.3 K | train | 0    
6 | decoder_dropout  | Dropout | 0      | train | 0    
7 | output_layer     | Linear  | 1.9 K  | train | 0    
-------------------------------------------------------------
155 K     Trainable params
0         Non-trainable params
155 K     Total params
0.621     Total estimated model params size (MB)
8         Modules in train mode
0         Modules in eval mode
0         Total Flops


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_epochs=50` reached.


🏃 View run magnificent-koi-279 at: http://localhost:5050/#/experiments/2/runs/02dff70a412542e4ae4b0f2f3aae8f45
🧪 View experiment at: http://localhost:5050/#/experiments/2


## Évaluation - Détection des attaques réalistes
Les anomalies sont détectées quand l'erreur de reconstruction dépasse un seuil optimal.

In [13]:
# Calcul des erreurs de reconstruction sur le jeu de test
model.eval()
model.to(device)

reconstruction_errors = []
all_targets = []

with torch.no_grad():
    for batch_x, batch_y in datamodule.test_dataloader():
        batch_x = batch_x.to(device)
        x_hat, _, _ = model(batch_x)

        mse = torch.mean((batch_x - x_hat) ** 2, dim=(1, 2))
        reconstruction_errors.extend(mse.cpu().numpy())
        all_targets.extend(batch_y.numpy())

reconstruction_errors = np.array(reconstruction_errors)
all_targets = np.array(all_targets)

print(f"Erreur moyenne (normales): {reconstruction_errors[all_targets == 0].mean():.4f}")
print(f"Erreur moyenne (attaques): {reconstruction_errors[all_targets == 1].mean():.4f}")
print(f"Ratio: {reconstruction_errors[all_targets == 1].mean() / reconstruction_errors[all_targets == 0].mean():.2f}x")

Erreur moyenne (normales): 0.0364
Erreur moyenne (attaques): 12.6773
Ratio: 348.08x


In [14]:
# Seuil optimal via precision-recall curve
precision, recall, thresholds = precision_recall_curve(all_targets, reconstruction_errors)
f1_scores = 2 * (precision * recall) / (precision + recall + 1e-8)
optimal_idx = np.argmax(f1_scores)
optimal_threshold = thresholds[optimal_idx] if optimal_idx < len(thresholds) else thresholds[-1]

print(f"Seuil optimal: {optimal_threshold:.4f}")
print(f"F1-score optimal: {f1_scores[optimal_idx]:.4f}")

all_preds = (reconstruction_errors > optimal_threshold).astype(int)

print("\n" + "=" * 50)
print(classification_report(all_targets, all_preds, target_names=['Normal', 'Attaque']))

auc = roc_auc_score(all_targets, reconstruction_errors)
print(f"AUC-ROC: {auc:.4f}")

Seuil optimal: 0.0031
F1-score optimal: 0.9096

              precision    recall  f1-score   support

      Normal       0.00      0.00      0.00       267
     Attaque       0.83      1.00      0.91      1344

    accuracy                           0.83      1611
   macro avg       0.42      0.50      0.45      1611
weighted avg       0.70      0.83      0.76      1611

AUC-ROC: 0.6049


In [15]:
cm = confusion_matrix(all_targets, all_preds)
tn, fp, fn, tp = cm.ravel()

print("Matrice de confusion:")
print(f"  TN={tn}  FP={fp}")
print(f"  FN={fn}  TP={tp}")
print(f"\nTaux de faux positifs: {fp/(fp+tn)*100:.2f}%")
print(f"Taux de détection (recall): {tp/(tp+fn)*100:.2f}%")

Matrice de confusion:
  TN=0  FP=267
  FN=1  TP=1343

Taux de faux positifs: 100.00%
Taux de détection (recall): 99.93%


In [16]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.metrics import roc_curve

fig = make_subplots(rows=1, cols=2, subplot_titles=(
    'Distribution des erreurs de reconstruction',
    'Courbe ROC'
))

fig.add_trace(go.Histogram(
    x=reconstruction_errors[all_targets == 0],
    name='Normal', opacity=0.7, nbinsx=50
), row=1, col=1)

fig.add_trace(go.Histogram(
    x=reconstruction_errors[all_targets == 1],
    name='Attaque', opacity=0.7, nbinsx=50
), row=1, col=1)

fig.add_vline(x=optimal_threshold, line_dash="dash", line_color="red", row=1, col=1)

fpr, tpr, _ = roc_curve(all_targets, reconstruction_errors)
fig.add_trace(go.Scatter(x=fpr, y=tpr, name=f'ROC (AUC={auc:.3f})'), row=1, col=2)
fig.add_trace(go.Scatter(x=[0, 1], y=[0, 1], line=dict(dash='dash'), name='Random'), row=1, col=2)

fig.update_layout(height=400, title_text="Évaluation du VAE-RNN sur attaques réalistes", barmode='overlay')
fig.update_xaxes(title_text="Erreur de reconstruction", row=1, col=1)
fig.update_xaxes(title_text="Taux de faux positifs", row=1, col=2)
fig.update_yaxes(title_text="Taux de vrais positifs", row=1, col=2)
fig.show()

In [17]:
import json

models_dir = Path.cwd().parent / "models"
models_dir.mkdir(exist_ok=True)

best_model_path = checkpoint.best_model_path
print(f"Meilleur modèle sauvegardé: {best_model_path}")

config = {
    'model': 'RnnVae',
    'threshold': float(optimal_threshold),
    'n_features': n_features,
    'seq_len': seq_len,
    'hidden_dim': HIDDEN_DIM,
    'latent_dim': LATENT_DIM,
    'n_layers': N_LAYERS,
    'kl_weight': KL_WEIGHT,
    'auc_roc': float(auc),
    'f1_score': float(f1_scores[optimal_idx]),
    'attacks_used': ['spoofing', 'masquerade', 'replay', 'suspension'],
}

with open(models_dir / 'rnn_vae_config.json', 'w') as f:
    json.dump(config, f, indent=2)

print(f"Configuration sauvegardée: {models_dir / 'rnn_vae_config.json'}")

Meilleur modèle sauvegardé: /Users/nikova/Desktop/canLock/CANlock/models/rnn_vae-epoch=44-val_loss=0.0388.ckpt
Configuration sauvegardée: /Users/nikova/Desktop/canLock/CANlock/models/rnn_vae_config.json


## Visualisation - Exemples de reconstruction
Observer le comportement du modèle sur quelques séquences normales vs attaques.

In [18]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Séparer les erreurs par type
normal_mask = all_targets == 0
attack_mask = all_targets == 1

# Prendre quelques exemples
# 3 normales (faible erreur), 3 attaques (haute erreur), 3 attaques (basse erreur = ratées)
normal_indices = np.where(normal_mask)[0]
attack_indices = np.where(attack_mask)[0]

# Trier par erreur
normal_sorted = normal_indices[np.argsort(reconstruction_errors[normal_indices])]
attack_sorted = attack_indices[np.argsort(reconstruction_errors[attack_indices])]

print('=== Normales (erreurs les plus basses) ===')
for idx in normal_sorted[:3]:
    print(f'  Index {idx}: erreur = {reconstruction_errors[idx]:.4f}')

print('\n=== Attaques détectées (erreurs les plus hautes) ===')
for idx in attack_sorted[-3:]:
    print(f'  Index {idx}: erreur = {reconstruction_errors[idx]:.4f}')

print('\n=== Attaques RATÉES (erreurs les plus basses = subtiles) ===')
for idx in attack_sorted[:3]:
    print(f'  Index {idx}: erreur = {reconstruction_errors[idx]:.4f}')

=== Normales (erreurs les plus basses) ===
  Index 56: erreur = 0.0037
  Index 44: erreur = 0.0044
  Index 151: erreur = 0.0046

=== Attaques détectées (erreurs les plus hautes) ===
  Index 1479: erreur = 1003.6038
  Index 1591: erreur = 1304.0177
  Index 751: erreur = 1307.2612

=== Attaques RATÉES (erreurs les plus basses = subtiles) ===
  Index 1574: erreur = 0.0031
  Index 1526: erreur = 0.0035
  Index 973: erreur = 0.0037


In [19]:
# Visualiser original vs reconstruction pour 1 normale, 1 attaque détectée, 1 attaque ratée
examples = {
    'Normale': normal_sorted[0],
    'Attaque détectée': attack_sorted[-1],
    'Attaque ratée (subtile)': attack_sorted[0],
}

model.eval()
model.to(device)

fig = make_subplots(
    rows=3, cols=1,
    subplot_titles=[f"{name} (erreur: {reconstruction_errors[idx]:.4f})" for name, idx in examples.items()],
    vertical_spacing=0.08,
)

for row, (name, idx) in enumerate(examples.items(), 1):
    x_orig = X_test[idx]  # (seq_len, n_features)
    x_tensor = torch.FloatTensor(x_orig).unsqueeze(0).to(device)
    with torch.no_grad():
        x_hat, _, _ = model(x_tensor)
    x_recon = x_hat.squeeze(0).cpu().numpy()
    
    # Afficher le premier feature (SPN)
    fig.add_trace(go.Scatter(y=x_orig[:, 0], name=f'Original', line=dict(color='blue'), showlegend=(row==1)), row=row, col=1)
    fig.add_trace(go.Scatter(y=x_recon[:, 0], name=f'Reconstruction', line=dict(color='red', dash='dash'), showlegend=(row==1)), row=row, col=1)

fig.update_layout(height=700, title_text='Original vs Reconstruction (1er SPN)')
fig.show()

In [20]:
# Heatmap: erreur de reconstruction par feature pour les 3 exemples
fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=[f"{name}" for name in examples.keys()],
)

for col, (name, idx) in enumerate(examples.items(), 1):
    x_orig = X_test[idx]
    x_tensor = torch.FloatTensor(x_orig).unsqueeze(0).to(device)
    with torch.no_grad():
        x_hat, _, _ = model(x_tensor)
    x_recon = x_hat.squeeze(0).cpu().numpy()
    
    # Erreur par (timestep, feature)
    err = (x_orig - x_recon) ** 2
    
    fig.add_trace(go.Heatmap(
        z=err.T,
        colorscale='Reds',
        showscale=(col == 3),
    ), row=1, col=col)
    fig.update_xaxes(title_text='Timestep', row=1, col=col)
    fig.update_yaxes(title_text='Feature', row=1, col=col)

fig.update_layout(height=350, title_text='Heatmap erreur de reconstruction (rouge = haute erreur)')
fig.show()